# RPT API Playground

**Setup:** Replace `YOUR_TOKEN_HERE` in the cell below with the token you received. Do not commit this notebook after adding your token.

In [10]:
import requests
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from dotenv import load_dotenv
import os

print("cwd:", os.getcwd())
load_dotenv(override=True)
print("token:", os.getenv("RPT_TOKEN"))

API_TOKEN = os.getenv("RPT_TOKEN")
API_URL   = os.getenv("RPT_API_URL")

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

cwd: /Users/i589033/Documents/SAP/RPT-1/rpt_playground
token: bfm_fb0993b1bfe3be929cfe0dc1be2561c03fe16645d999aa1539f0d8acde150470


## Data Generation

We generate a synthetic dataset with two features `x1`, `x2` and a target `y` defined by a polynomial function.

In [ ]:
coef = {"a": 1, "b": -1, "c": 0.0, "d": 2.0, "e": -0.5, "f": 0.1}
rng = np.random.default_rng(42)
N = 1000

x1 = rng.uniform(-2, 2, size=N)
x2 = rng.uniform(-2, 2, size=N)
y = (
    coef["a"] * x1**2
    + coef["b"] * x2**2
    + coef["c"] * x1 * x2
    + coef["d"] * x1
    + coef["e"] * x2
    + coef["f"]
)

df = pd.DataFrame({
    "id": [f"row_{i}" for i in range(N)],
    "x1": x1.round(4),
    "x2": x2.round(4),
    "y":  y.round(4)
})
df.head()

## API Helpers

The RPT API takes a list of rows. Context rows have known `y` values; eval rows have `y = "[PREDICT]"`. The model predicts the missing values using the context as in-context examples.

In [ ]:
import json

def get_payload(context_df, eval_df_batch):
    context_rows = json.loads(context_df.to_json(orient="records"))
    eval_df_copy = eval_df_batch.copy()
    eval_df_copy["y"] = "[PREDICT]"
    eval_rows = json.loads(eval_df_copy.to_json(orient="records"))
    return {"rows": context_rows + eval_rows, "index_column": "id"}

def get_response(api_url, headers, payload):
    response = requests.post(api_url, headers=headers, json=payload, timeout=30)
    if not response.ok:
        print(f"Status {response.status_code}: {response.text}")
    response.raise_for_status()
    return response.json()

def get_predictions(result, target_col="y"):
    df_preds = pd.json_normalize(
        result["prediction"]["predictions"],
        record_path=target_col
    )
    return df_preds["prediction"].astype(float).values

def evaluate_full_dataset(context_df, eval_df, api_url, headers, batch_size=25):
    y_true_all, y_pred_all = [], []
    for start in range(0, len(eval_df), batch_size):
        batch_df = eval_df.iloc[start:start + batch_size]
        payload = get_payload(context_df, batch_df)
        result = get_response(api_url, headers, payload)
        y_pred_all.extend(get_predictions(result))
        y_true_all.extend(batch_df["y"].values)
    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    return {
        "MSE": mean_squared_error(y_true_all, y_pred_all),
        "R2": r2_score(y_true_all, y_pred_all),
        "y_true": y_true_all,
        "y_pred": y_pred_all
    }

## Predict

In [ ]:
print("df columns:", df.columns.tolist())
print("first row:", df.iloc[0].to_dict())

payload = get_payload(df[25:35], df[:1])
print("\nfirst payload row:", payload["rows"][0])
print("last payload row:", payload["rows"][-1])

## Sandbox

Use this cell to experiment freely.